In [0]:
def test_netflix_mom_growth(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, sum as _sum, lag, to_date, date_format, round

    data = [
        ("U1", "2024-01-01", 100),
        ("U1", "2024-02-01", 300)
    ]

    columns = ["user_id", "watch_date", "watch_minutes"]
    df = spark.createDataFrame(data, columns) \
              .withColumn("watch_date", to_date("watch_date"))

    df = df.withColumn("month", date_format("watch_date", "yyyy-MM"))

    monthly_df = df.groupBy("user_id", "month") \
                   .agg(_sum("watch_minutes").alias("monthly_watch_time"))

    window_spec = Window.partitionBy("user_id").orderBy("month")

    result_df = monthly_df.withColumn("prev_month_watch", lag("monthly_watch_time").over(window_spec)) \
        .withColumn(
        "mom_growth_pct",
        round(((col("monthly_watch_time") - col("prev_month_watch")) / col("prev_month_watch")) * 100, 2)
        )

    result = [(r["user_id"], r["month"], r["mom_growth_pct"]) for r in result_df.collect()]

    try:
        assert result[1][2] == 200.0
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected 200.0 but got", result[1][2])

test_netflix_mom_growth(spark)